In [2]:
import numpy as np 
import pandas as pd

In [ ]:
df = pd.read_excel('STL_Label_2025-05-06_IPI.xlsx', header=None)

In [ ]:
df.shape

In [ ]:
df.head(7)

In [ ]:
# Use row 3 as columns
new_header = df.iloc[3].tolist()  # convert to list to avoid weird index name
df.columns = new_header

# Drop first four rows (0 to 3)
df = df.drop([0, 1, 2, 3])

# Reset index
df = df.reset_index(drop=True)

In [ ]:
df

In [ ]:
cols = df.columns.tolist()
cols[0] = "filename"     # first column
cols[-1] = "filetype"    # last column
df.columns = cols

In [ ]:
df = df.fillna(0)

In [ ]:
df

In [ ]:
df.columns

In [ ]:
df.columns = [int(col) if isinstance(col, float) or (isinstance(col, str) and col.replace('.', '', 1).isdigit() and float(col).is_integer()) else col for col in df.columns]


In [ ]:
# for upper jaw
cols_to_process = [18, 17, 16, 15, 14, 13, 12, 11, 21, 22, 23, 24, 25, 26, 27, 28]
def to_binary(val):
    # Convert val to string and strip spaces
    val_str = str(val).strip()
    # If the whole string is '0' or '0.0', consider zero, else 1
    if val_str == '0' or val_str == '0.0' or val_str == 0:
        return 0
    else:
        return 1

df[cols_to_process] = df[cols_to_process].applymap(to_binary).astype(int)

In [ ]:
# for lower jaw
cols_to_process = [38,37,36,35,34,33,32,31,41,42,43,44,45,46,47,48]
def to_binary(val):
    # Convert val to string and strip spaces
    val_str = str(val).strip()
    # If the whole string is '0' or '0.0', consider zero, else 1
    if val_str == '0' or val_str == '0.0' or val_str == 0:
        return 0
    else:
        return 1

df[cols_to_process] = df[cols_to_process].applymap(to_binary).astype(int)  # Example to check the type conversion of a specific cell

In [ ]:
df.head(7)

In [ ]:
df['new_id'] = (
    df['filename']
    .str.strip()                              # remove extra spaces
    .str.replace(' ', '_', regex=False)       # replace spaces with underscores
    .str.replace('-', '_', regex=False)       # replace hyphens with underscores
    .str.replace('LowerJawScan', 'lower', regex=False)
    .str.replace('UpperJawScan', 'upper', regex=False)
)


In [ ]:
df.to_csv("label_processed.csv", index=False)
print("\nSaved labeled test image data to label_processed.csv")

In [ ]:

print(df.filename.nunique(), df.new_id.nunique())

In [ ]:
df.columns

In [ ]:
df.head(5)

In [ ]:
fdi_columns = [
    18, 17, 16, 15, 14, 13, 12, 11, 21, 22, 23, 24, 25, 26, 27, 28,
    38, 37, 36, 35, 34, 33, 32, 31, 41, 42, 43, 44, 45, 46, 47, 48
]

# Ensure you only try to flip columns that actually exist in the DataFrame
columns_to_flip = [col for col in fdi_columns if col in df.columns]

# --- Label Flipping Operation ---
# Apply the 1 - x operation to each tooth column.
# This converts 1s to 0s and 0s to 1s.
df[columns_to_flip] = 1 - df[columns_to_flip]

# You can verify the change by inspecting a few rows of the tooth columns
print("Labels have been flipped. Displaying the first 5 rows of tooth data:")
print(df.loc[:4, columns_to_flip])


df.to_csv("label_flipped.csv", index=False)
print("\nSaved labeled test image data to label_flipped.csv")

In [ ]:
df= pd.read_csv("label_flipped.csv")

In [ ]:
# Drop rows 123 and 124
df = df.drop([123, 124]).reset_index(drop=True)

# Save back to the same file
df.to_csv("label_flipped.csv", index=False)

In [ ]:
df[df.duplicated('new_id', keep=False)][['filename', 'new_id']].sort_values('new_id')

In [ ]:
duplicate_filenames = df[df['new_id'].duplicated(keep=False)]

In [ ]:
duplicate_filenames.shape

In [ ]:
duplicate_filenames

In [ ]:
# import os

# # Source folder
# folder_path = "/home/user/tbrighton/blender_outputs/parsed_ply"

# # List all files in the folder
# files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]

# for f in files:
#     old_path = os.path.join(folder_path, f)
    
#     # Apply the same transformation as your df['new_id']
#     new_name = (
#         f.strip()                              # remove extra spaces
#         .replace(' ', '_')                     # replace spaces with underscores
#         .replace('-', '_')                     # replace hyphens with underscores
#         .replace('LowerJawScan', 'lower')
#         .replace('UpperJawScan', 'upper')
#     )
    
#     new_path = os.path.join(folder_path, new_name)
    
#     # Rename the file
#     os.rename(old_path, new_path)
#     print(f"Renamed: {f} -> {new_name}")


In [ ]:
import os

folder_path = "/home/user/tbrighton/blender_outputs/parsed_ply"

# All files in the folder
folder_files = set(os.listdir(folder_path))

# Add .ply to new_id for comparison
new_ids_with_ext = set(df['new_id'].astype(str) + ".ply")

# Find matching files
matching_files = folder_files.intersection(new_ids_with_ext)

print(f"Number of matching files: {len(matching_files)}")
#print("Matching files:", matching_files)


In [ ]:
### Total 174 files were imported from 190 files, after parsing the name the duplicate names were not copied over.
### Matched 171 files with the new naming scheme 

In [ ]:
import re

def extract_id(row):
    # extract digits at the start (before first _ or -)
    match = re.match(r"^(\d+)", row)
    if not match:
        return row  # fallback: return as-is if no match
    
    file_id = match.group(1)
    
    if "UpperJawScan" in row:
        return f"{file_id}_upper"
    elif "LowerJawScan" in row:
        return f"{file_id}_lower"
    else:
        return file_id

df["new_id"] = df["filename"].apply(extract_id)


In [ ]:
df

In [ ]:
import os
import re

folder = "/home/user/tbrighton/blender_outputs/test_ply_views"

for fname in os.listdir(folder):
    if fname.endswith(".png"):
        old_path = os.path.join(folder, fname)

        # extract digits at the start (before first '_' or '-')
        match = re.match(r"^(\d+)", fname)
        if not match:
            continue  # skip if no id found
        file_id = match.group(1)

        # decide upper/lower based on filename
        lower_fname = fname.lower()
        if "upper" in lower_fname:
            new_name = f"{file_id}_upper.png"
        elif "lower" in lower_fname:
            new_name = f"{file_id}_lower.png"
        else:
            new_name = f"{file_id}.png"  # fallback if neither found

        new_path = os.path.join(folder, new_name)

        os.rename(old_path, new_path)
        print(f"Renamed: {fname} → {new_name}")


In [ ]:
import os

# get all .png names from folder (without extension)
folder = "/home/user/tbrighton/blender_outputs/test_ply_views"
png_names = [os.path.splitext(f)[0] for f in os.listdir(folder) if f.endswith(".png")]

# get new_id column from dataframe
df_names = df["new_id"].tolist()

# convert to sets for matching
set_png = set(png_names)
set_df = set(df_names)

# intersection = matching names
matches = set_df.intersection(set_png)

print(f"Total in df: {len(set_df)}")
print(f"Total in folder: {len(set_png)}")
print(f"Matching: {len(matches)}")

# if you want the actual matching names:
#print("Matching IDs:", matches)

# if you want the ones missing from either side:
missing_in_folder = set_df - set_png
missing_in_df = set_png - set_df


In [3]:
df_new=pd.read_csv("label_flipped.csv")

In [ ]:
df.columns

In [ ]:
df_new.to_excel("label_flipped.xlsx", index=False)